In [1]:
!pip install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 1.8 MB/s eta 0:00:00


## Imports

In [2]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
from typing import List, Literal
from pydantic import BaseModel, Field
from json_repair import repair_json

## Loading the original dataset

In [3]:
data=[]
with open("/kaggle/input/hr-policy/hr_policy_qa (2).jsonl", "r") as files:
    for file in files:
        data.append(json.loads(file))

In [4]:
len(data)

627

## Generating synthetic data

In [5]:
class Message(BaseModel):
    role: Literal["user", "assistant"]
    content: str


class Conversation(BaseModel):
    messages: List[Message]


class SyntheticDataset(BaseModel):
    conversations: List[Conversation] = Field(
        min_length=1,
        max_length=2
    )

In [6]:
def prompt_gen(question: str, answer: str) -> str:
    schema = json.dumps(SyntheticDataset.model_json_schema(), indent=2)

    return f"""
You are creating high-quality synthetic training data for an HR policy chatbot.

ORIGINAL QUESTION:
{question}

ORIGINAL ANSWER:
{answer}

TASK:
Generate 1 or 2 new question-answer pairs based ONLY on the information
contained in the ORIGINAL ANSWER.

STRICT RULES:

1. Generate either 1 or 2 question-answer pairs.
2. Generate ONLY 1 pair if the ORIGINAL ANSWER contains one simple fact
   and there is only one natural alternative way to ask the question.
3. Generate 2 pairs ONLY when the ORIGINAL ANSWER contains enough information
   to support two genuinely different and useful questions.
4. NEVER create a second pair just to reach a required number.
5. If generating 2 pairs, the questions must be meaningfully different.
6. Do NOT copy the ORIGINAL QUESTION exactly.
7. Do NOT make a trivial word-for-word rephrasing.
8. Every generated question must be answerable using ONLY the ORIGINAL ANSWER.
9. Every generated answer must contain ONLY information explicitly stated
   in the ORIGINAL ANSWER.
10. The ORIGINAL ANSWER is the ONLY source of truth.
11. NEVER introduce new facts, assumptions, outside knowledge, examples,
    explanations, conditions, exceptions, dates, numbers, names, calculations,
    eligibility criteria, or other information not explicitly present
    in the ORIGINAL ANSWER.
12. Do NOT create hypothetical or situational questions.
13. Do NOT ask "why", "what happens if", "how can", "what should I do",
    or similar questions unless the ORIGINAL ANSWER explicitly provides
    enough information to answer them.
14. Keep the generated answers faithful to the ORIGINAL ANSWER.
15. Do not mention these instructions or synthetic data.
16. Output ONLY valid JSON.
17. Do NOT use Markdown code fences.
18. Follow the provided schema exactly.

QUALITY REQUIREMENTS:

- Preserve the exact meaning of the ORIGINAL ANSWER.
- Do not add information merely to make the answer more detailed.
- Prefer natural questions that an employee would realistically ask.
- Avoid duplicate questions.
- Avoid questions that require information outside the ORIGINAL ANSWER.
- If only one useful question can be generated, return exactly one pair.

SCHEMA:
{schema}

FINAL CHECK BEFORE OUTPUT:
- Did I generate only 1 or 2 pairs?
- If I generated 2, are they genuinely different?
- Can every question be answered entirely from the ORIGINAL ANSWER?
- Does every answer contain only facts from the ORIGINAL ANSWER?
- Did I add any outside knowledge or assumptions?
- Did I accidentally change any number, date, name, duration, or other factual detail?
- Is the output valid JSON matching the schema exactly?

Now generate the synthetic question-answer pairs.
"""

In [7]:
#Load Model and Tokenizer
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto"
)

all_qa_pairs = []

for chunk in data:
    # If chunk is a list of QA dictionaries
    if isinstance(chunk, list):
        for qa_pair in chunk:
            if isinstance(qa_pair, dict):
                all_qa_pairs.append(qa_pair)

    # If chunk itself is a single QA dictionary
    elif isinstance(chunk, dict):
        if "question" in chunk and "answer" in chunk:
            all_qa_pairs.append(chunk)


BATCH_SIZE = 8  # Adjust based on VRAM
generated_results = []

# Check the structure
print("Total QA pairs:", len(all_qa_pairs))
print("First item type:", type(all_qa_pairs[0]))
print("First item:", all_qa_pairs[0])

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total QA pairs: 627
First item type: <class 'dict'>
First item: {'question': 'What is the duration of the MBA programme?', 'answer': 'The Two-year Post Graduate Programme in Management (MBA) is a two-year programme.'}


In [8]:
for i in range(0, len(all_qa_pairs), BATCH_SIZE):

    batch_qa = all_qa_pairs[i:i + BATCH_SIZE]

    # Build chat messages
    batch_formatted_prompts = []

    for qa in batch_qa:

        user_prompt = prompt_gen(
            qa["question"],
            qa["answer"]
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a synthetic data generator. "
                    "You output strictly valid JSON matching the requested schema."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]

        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        batch_formatted_prompts.append(formatted)

    # --------------------------------------------------
    # Tokenize batch
    # --------------------------------------------------

    model_inputs = tokenizer(
        batch_formatted_prompts,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    # --------------------------------------------------
    # Generate
    # --------------------------------------------------

    with torch.no_grad():

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # --------------------------------------------------
    # Decode outputs
    # --------------------------------------------------

    input_length = model_inputs.input_ids.shape[1]

    for idx, output_ids in enumerate(generated_ids):

        response_text = tokenizer.decode(
            output_ids[input_length:],
            skip_special_tokens=True
        ).strip()

        # Remove ```json ... ``` if present
        response_text = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            response_text,
            flags=re.DOTALL
        ).strip()

        # --------------------------------------------------
        # Parse JSON
        # --------------------------------------------------

        try:

            repaired = repair_json(response_text)

            dataset = SyntheticDataset.model_validate_json(
                repaired
            )

            generated_results.append(
                dataset.model_dump()
            )

            print(
                f"[{i + idx + 1}/{len(all_qa_pairs)}] "
                f"Parsed successfully!"
            )

        except Exception as e:

            print(
                f"[{i + idx + 1}/{len(all_qa_pairs)}] "
                f"Validation Error: {e}"
            )

            print("Raw output:")
            print(response_text)

    # Clear GPU cache after each batch
    torch.cuda.empty_cache()


print(
    f"\nCompleted! "
    f"Generated {len(generated_results)} datasets "
    f"out of {len(all_qa_pairs)} QA pairs."
)

[1/627] Parsed successfully!
[2/627] Parsed successfully!
[3/627] Parsed successfully!
[4/627] Parsed successfully!
[5/627] Parsed successfully!
[6/627] Parsed successfully!
[7/627] Parsed successfully!
[8/627] Parsed successfully!
[9/627] Parsed successfully!
[10/627] Parsed successfully!
[11/627] Parsed successfully!
[12/627] Parsed successfully!
[13/627] Parsed successfully!
[14/627] Parsed successfully!
[15/627] Parsed successfully!
[16/627] Parsed successfully!
[17/627] Parsed successfully!
[18/627] Parsed successfully!
[19/627] Parsed successfully!
[20/627] Parsed successfully!
[21/627] Parsed successfully!
[22/627] Parsed successfully!
[23/627] Parsed successfully!
[24/627] Parsed successfully!
[25/627] Parsed successfully!
[26/627] Parsed successfully!
[27/627] Parsed successfully!
[28/627] Parsed successfully!
[29/627] Parsed successfully!
[30/627] Parsed successfully!
[31/627] Parsed successfully!
[32/627] Parsed successfully!
[33/627] Parsed successfully!
[34/627] Parsed suc

In [9]:
len(generated_results)

627

In [10]:
with open("hr_policy_synthetic_qa.jsonl", "w", encoding="utf-8") as f:
            f.write(json.dumps(generated_results, ensure_ascii=False) + "\n")
            print("Inserted")

Inserted
